# Klasifikasi Kacamata dan Helm Menggunakan GLCM + HOG + KNN
Notebook ini mengimplementasikan klasifikasi menggunakan kombinasi ekstraksi fitur **GLCM (tekstur)** dan **HOG (bentuk)**, kemudian diklasifikasikan menggunakan **K-Nearest Neighbors (KNN)** dengan nilai K optimal yang dicari secara otomatis.

In [ ]:
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from skimage.feature import graycomatrix, graycoprops, hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

IMG_SIZE = (128, 128)

def extract_features(image_path):
    """Mengekstrak fitur gabungan GLCM (tekstur) + HOG (bentuk) dari sebuah gambar."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    # Resize gambar agar dimensi seragam
    img_resized = cv2.resize(img, IMG_SIZE)
    
    # --- Fitur Tekstur: GLCM (Gray Level Co-occurrence Matrix) ---
    distances = [1, 3]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(img_resized, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)
    contrast     = graycoprops(glcm, 'contrast').flatten()
    correlation  = graycoprops(glcm, 'correlation').flatten()
    energy       = graycoprops(glcm, 'energy').flatten()
    homogeneity  = graycoprops(glcm, 'homogeneity').flatten()
    asm          = graycoprops(glcm, 'ASM').flatten()
    dissimilarity= graycoprops(glcm, 'dissimilarity').flatten()
    glcm_features = np.hstack([contrast, correlation, energy, homogeneity, asm, dissimilarity])
    
    # --- Fitur Bentuk: HOG (Histogram of Oriented Gradients) ---
    hog_features = hog(
        img_resized,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        visualize=False
    )
    
    # --- Gabungkan GLCM + HOG menjadi satu vektor fitur ---
    return np.hstack([glcm_features, hog_features])

def load_dataset(base_dir):
    """Memuat semua gambar dari direktori dataset dan mengekstrak fiturnya."""
    features_list = []
    labels_list = []
    classes = ['helm', 'kacamata']
    
    for class_idx, class_name in enumerate(classes):
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Direktori tidak ditemukan: {class_dir}")
            continue
        files = glob.glob(os.path.join(class_dir, '*.*'))
        print(f"Memproses kelas '{class_name}': {len(files)} gambar ditemukan...")
        for img_path in tqdm(files, desc=f'Ekstraksi {class_name}'):
            feat = extract_features(img_path)
            if feat is not None:
                features_list.append(feat)
                labels_list.append(class_idx)
    
    return np.array(features_list), np.array(labels_list)

# Muat dataset training
print('Memuat dataset training (GLCM + HOG)...')
X_raw, y_raw = load_dataset('dataset/train')
print(f'\nSelesai! Jumlah sampel: {X_raw.shape[0]}, Dimensi fitur: {X_raw.shape[1]}')


## Mencari Nilai K Optimal
Kita bagi data training menjadi 80% untuk melatih dan 20% untuk validasi. Kemudian kita coba berbagai nilai K untuk menemukan yang menghasilkan akurasi validasi terbaik.

In [ ]:
# Split 80:20 untuk mencari K optimal
X_tr, X_val, y_tr, y_val = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Standarisasi
scaler_val = StandardScaler()
X_tr_scaled  = scaler_val.fit_transform(X_tr)
X_val_scaled = scaler_val.transform(X_val)

print(f'Data Training: {X_tr_scaled.shape[0]} sampel')
print(f'Data Validasi: {X_val_scaled.shape[0]} sampel')
print('\nMencari nilai K optimal...')

k_values = list(range(1, 16, 2))
val_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_scaled, y_tr)
    preds = knn.predict(X_val_scaled)
    acc = accuracy_score(y_val, preds)
    val_accuracies.append(acc)
    print(f'  k = {k:2d}  ->  Akurasi Validasi: {acc*100:.2f}%')

# Visualisasi K vs Akurasi
plt.figure(figsize=(10, 5))
plt.plot(k_values, val_accuracies, marker='o', linestyle='-', color='#1e88e5', linewidth=2.5, markersize=8)
plt.title('Nilai K vs Akurasi Validasi (GLCM + HOG)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Nilai K (Jumlah Tetangga)', fontsize=12)
plt.ylabel('Akurasi Validasi', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(k_values)
for i, acc in enumerate(val_accuracies):
    plt.annotate(f'{acc:.4f}', (k_values[i], acc), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
plt.ylim(min(val_accuracies) - 0.05, max(val_accuracies) + 0.05)
plt.tight_layout()
plt.show()

best_k = k_values[int(np.argmax(val_accuracies))]
print(f'\nNilai K optimal: {best_k} (Akurasi Validasi: {max(val_accuracies)*100:.2f}%)')


## Pelatihan Model Final
Model final dilatih menggunakan **seluruh data training** (tanpa split) dengan nilai K terbaik yang sudah ditemukan.

In [ ]:
# Standarisasi menggunakan seluruh data training
scaler_final = StandardScaler()
X_train_final = scaler_final.fit_transform(X_raw)

# Training model final dengan K terbaik
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_final, y_raw)

print(f'Model KNN Final (k={best_k}) dengan fitur GLCM + HOG berhasil dilatih!')


## Demo Klasifikasi: Visualisasi Tahapan Preprocessing
Menampilkan setiap tahapan preprocessing untuk satu contoh gambar dari masing-masing kelas.

In [ ]:
def classify_and_visualize(image_path, model, scaler_model):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return 'Error'

    h_orig, w_orig = img_bgr.shape[:2]
    img_gray_orig  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_resized    = cv2.resize(img_gray_orig, IMG_SIZE)

    _, hog_image = hog(
        img_resized, orientations=8, pixels_per_cell=(16, 16),
        cells_per_block=(2, 2), visualize=True
    )

    feat = extract_features(image_path)
    feat_scaled = scaler_model.transform([feat])
    pred_id = model.predict(feat_scaled)[0]
    proba   = model.predict_proba(feat_scaled)[0]
    pred_label = 'Helm' if pred_id == 0 else 'Kacamata'
    confidence = proba[pred_id] * 100

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # 1. Gambar Asli
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'1. Gambar Asli\n({w_orig}x{h_orig})')

    # 2. Resize
    axes[1].imshow(cv2.cvtColor(cv2.resize(img_bgr, IMG_SIZE), cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'2. Hasil Resize\n({IMG_SIZE[0]}x{IMG_SIZE[1]})')

    # 3. Grayscale
    axes[2].imshow(img_resized, cmap='gray')
    axes[2].set_title(f'3. Grayscale\n({IMG_SIZE[0]}x{IMG_SIZE[1]})')

    # 4. HOG + Prediksi
    axes[3].imshow(hog_image, cmap='gray')
    axes[3].set_title(f'4. HOG Features\nKNN: {pred_label} ({confidence:.1f}%)')

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return pred_label

print('=== DEMO KLASIFIKASI ===')
helm_samples = glob.glob('dataset/test/helm/*.*')
if helm_samples:
    classify_and_visualize(helm_samples[0], knn_final, scaler_final)

kaca_samples = glob.glob('dataset/test/kacamata/*.*')
if kaca_samples:
    classify_and_visualize(kaca_samples[0], knn_final, scaler_final)


## Evaluasi Model pada Dataset Test

In [ ]:
print('Memuat dataset test...')
X_test_raw, y_test = load_dataset('dataset/test')
print(f'Selesai! Jumlah sampel test: {X_test_raw.shape[0]}')

X_test_scaled = scaler_final.transform(X_test_raw)
y_pred = knn_final.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_pred)
print(f'\nAkurasi Model KNN + GLCM + HOG pada Dataset Test: {test_accuracy * 100:.2f}%')

labels = ['Helm', 'Kacamata']
print('\n--- CLASSIFICATION REPORT ---')
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - KNN + GLCM + HOG', fontsize=12, fontweight='bold')
plt.show()


## 🧪 Pengujian dengan Citra Baru (Custom Image)
Bagian ini digunakan untuk menguji model menggunakan gambar baru di luar dataset train/test.

In [ ]:
image_path = r"D:\Rezza\Gambar\Cuplikan Layar\Screenshot 2026-06-02 185948.png"

def predict_custom_image(image_path):
    if not os.path.exists(image_path):
        print('Error: Berkas gambar tidak ditemukan!')
        return

    # Ekstraksi Fitur GLCM + HOG dari gambar baru
    features = extract_features(image_path)
    if features is None:
        print('Error: Gagal mengekstrak fitur gambar.')
        return

    # Standardisasi
    features_scaled = scaler_final.transform([features])

    # Prediksi kelas dan probabilitas voting
    prediction  = knn_final.predict(features_scaled)[0]
    probabilities = knn_final.predict_proba(features_scaled)[0]

    classes = ['Helm', 'Kacamata']
    predicted_class = classes[prediction]
    confidence = probabilities[prediction] * 100

    # Membaca gambar asli untuk visualisasi
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Visualisasi Hasil
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    color = '#2e7d32' if prediction == 0 else '#1565c0'
    plt.title(f'Prediksi: {predicted_class.upper()} ({confidence:.2f}% Confidence)',
              fontsize=14, fontweight='bold', color=color)
    plt.axis('off')
    plt.show()

# Jalankan pengujian gambar baru
predict_custom_image(image_path)
